In [10]:
import duckdb
import os
import polars as pl
import json
from IPython.display import display
import yaml

from shared_code.database.directory_creation import *
from shared_code.database.database_and_schema_creation import *
from pipelines.modrinth_api_general.schemas.modrinth_general_schemas import *

In [11]:
def read_latest_raw_project_listings(bronze_db_path:str, modrinth_project_listings_table_name: str) -> pl.DataFrame:
    """
    Read the latest raw project listings from the bronze database.
    """
    query = f"""
        WITH latest_run AS (
            SELECT
                run_id,
                project_type
            FROM (
                SELECT
                    run_id,
                    project_type,
                    MAX(c_pull_timestamp_utc) AS max_pull_timestamp,
                    ROW_NUMBER() OVER (
                        PARTITION BY project_type
                        ORDER BY
                            MAX(c_pull_timestamp_utc) DESC,
                            run_id DESC
                    ) AS rn
                FROM {modrinth_project_listings_table_name}
                GROUP BY
                    run_id,
                    project_type
            )
            WHERE rn = 1
        )

        SELECT
            r.run_id,
            r.project_type,
            r.project_id,
            r.payload,
            r.c_pull_timestamp_utc
        FROM {modrinth_project_listings_table_name} r

        INNER JOIN latest_run l
            ON r.run_id = l.run_id
            AND r.project_type = l.project_type
    """

    with duckdb.connect(
        bronze_db_path,
        read_only=True,
    ) as bronze_con:
        return bronze_con.execute(query).pl()

In [12]:
def unpack_project_listings(raw_df: pl.DataFrame) -> pl.DataFrame:
    # everything outside payload is treated as authoritative metadata
    metadata_df = raw_df.drop("payload")

    payload_rows = [
        json.loads(payload)
        for payload in raw_df["payload"].to_list()
    ]

    payload_df = pl.from_dicts(payload_rows, infer_schema_length=None)

    # if a payload column already exists in metadata,
    # keep the metadata version and drop the payload version
    duplicate_columns = list(
        set(metadata_df.columns) & set(payload_df.columns)
    )

    if duplicate_columns:
        payload_df = payload_df.drop(duplicate_columns)

    return metadata_df.hstack(payload_df)

In [13]:
def write_polars_to_silver(
    silver_db_path: str,
    df: pl.DataFrame,
    table_name: str,
) -> None:
    with duckdb.connect(silver_db_path) as silver_con:
        silver_con.register("source_df", df)

        silver_con.execute(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *
            FROM source_df
        """)

        silver_con.unregister("source_df")

In [14]:
def create_base_api_project_listings(bronze_db_path: str, silver_db_path:str, table_name: str) -> None:

    raw_df = read_latest_raw_project_listings(bronze_db_path=bronze_db_path, modrinth_project_listings_table_name=table_name)
    print(f"Read in {raw_df.height:,} project listings from bronze.")
    raw_count_df = (
        raw_df
        .group_by(
            "run_id",
            "project_type",
        )
        .agg(
            pl.len().cast(pl.Int64).alias("row_count")
        )
    )
    total_row = pl.DataFrame({
    "run_id": [None],
    "project_type": ["total_listings"],
    "row_count": [raw_count_df["row_count"].sum()],
    })

    raw_count_df = pl.concat(
        [
            raw_count_df,
            total_row,
        ],
        how="vertical",
    )

    print(raw_count_df)

    silver_df = unpack_project_listings(raw_df)
    
    # with pl.Config(
    #     tbl_rows = 10,
    #     tbl_cols = 30):

    #         display(silver_df)
            

    # rules_df = read_data_rules(
    #     b1=b1,
    #     table_name=b1.base_api_project_listings_table_name,
    # )

    # silver_df = apply_data_rules(
    #     df=silver_df,
    #     rules_df=rules_df,
    # )

    write_polars_to_silver(silver_db_path=silver_db_path, df=silver_df, table_name=table_name)

    print(
        f"Silver project listings written: "
        f"{silver_df.height:,}"
    )

In [15]:
config_path = "stream-config.yml"
with open(config_path, "r") as file:
    config_data = yaml.safe_load(file)

In [16]:
env = 'dev'
project_root = os.path.abspath(
    os.path.join(
        os.getcwd(),
        "..",
        "..",
        "..",
    )
)
headers = config_data['main']['headers']
ingestion_log = config_data['main']['ingestion-log']

streams = config_data['streams']
source_url = streams[0]['source-url']
table_name = streams[0]['table-name']
search_limit = streams[0]['search-limit']
concurrency_limit = streams[0]['concurrency-limit']
bronze_path = os.path.join(project_root, config_data['main']['parent-folder'], 'bronze', env, build_db_filename(streams[0]['source-url']))
silver_path = os.path.join(project_root, config_data['main']['parent-folder'], 'silver', env, build_db_filename(streams[0]['source-url']))
print(bronze_path)
print(silver_path)

/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||api.modrinth.com.duckdb
/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/silver/dev/https:||api.modrinth.com.duckdb


In [ ]:
build_layer_directory(os.path.dirname(silver_path))
init_db(db_path=silver_path)
#dont need to initialize silver schemas becuase it is inferred from the polars dataframe when writing to silver

#bring clean silver data and enforce schema
create_base_api_project_listings(bronze_db_path=bronze_path, silver_db_path=silver_path, table_name=table_name)

Read in 158,739 project listings from bronze.
shape: (8, 3)
┌─────────────────────────────────┬───────────────────────┬───────────┐
│ run_id                          ┆ project_type          ┆ row_count │
│ ---                             ┆ ---                   ┆ ---       │
│ str                             ┆ str                   ┆ i64       │
╞═════════════════════════════════╪═══════════════════════╪═══════════╡
│ 8af5da09-b203-4d7e-bcab-e78002… ┆ resourcepack          ┆ 33431     │
│ 8af5da09-b203-4d7e-bcab-e78002… ┆ modpack               ┆ 18113     │
│ 8af5da09-b203-4d7e-bcab-e78002… ┆ minecraft_java_server ┆ 1972      │
│ 8af5da09-b203-4d7e-bcab-e78002… ┆ mod                   ┆ 73259     │
│ 8af5da09-b203-4d7e-bcab-e78002… ┆ plugin                ┆ 16878     │
│ 8af5da09-b203-4d7e-bcab-e78002… ┆ shader                ┆ 836       │
│ 8af5da09-b203-4d7e-bcab-e78002… ┆ datapack              ┆ 14250     │
│ null                            ┆ total_listings        ┆ 158739    │
└───

In [18]:
with duckdb.connect(silver_path) as silver_con:
    silver_df = silver_con.execute(f"""
        SELECT *
        FROM {table_name}
    """).pl()
    
with pl.Config(
        tbl_rows = 5,
        tbl_cols = 35):

            display(silver_df)

run_id,project_type,project_id,c_pull_timestamp_utc,all_project_types,slug,author,author_id,organization,organization_id,title,description,categories,display_categories,versions,downloads,follows,icon_url,date_created,date_modified,latest_version,license,client_side,server_side,environment,disclosure_types,gallery,featured_gallery,color
str,str,str,"datetime[μs, America/Los_Angeles]",list[str],str,str,str,str,str,str,str,list[str],list[str],list[str],i64,i64,str,str,str,str,str,str,str,list[str],list[str],list[str],str,i64
"""8af5da09-b203-4d7e-bcab-e78002…","""mod""","""Od4DEuiL""",2026-08-25 14:26:48.932118 PDT,"[""mod""]","""ghost-train-sound""","""TheSPGhost""","""Ttj8njDA""","""Ghost Workshop""","""qiQq62i4""","""Ghost Train Sound""","""I have added custom sounds for…","[""decoration"", ""fabric""]","[""decoration"", ""fabric""]","[""1.20.1""]",98,2,"""https://cdn.modrinth.com/data/…","""2026-02-17T21:14:52.323331+00:…","""2026-02-12T13:31:33.871448+00:…","""Gl0IRgX8""","""LicenseRef-Ghost-Train-Sound-C…","""required""","""required""","[""client_and_server""]","[""archived""]",[],null,2040608
"""8af5da09-b203-4d7e-bcab-e78002…","""mod""","""dYI6QlsF""",2026-08-25 14:26:48.932118 PDT,"[""mod""]","""freedom-chat""","""Lawnman893""","""RaXfFl6F""",null,null,"""Freedomchat""","""FreedomChat adds filtering to …","[""neoforge"", ""social""]","[""neoforge"", ""social""]","[""1.21.1""]",98,0,"""""","""2026-02-05T07:24:49.817296+00:…","""2026-02-06T19:49:58.636527+00:…","""Pl7hyZxx""","""MIT""","""unsupported""","""required""","[""server_only""]",[],[],null,null
"""8af5da09-b203-4d7e-bcab-e78002…","""mod""","""66zLdBvC""",2026-08-25 14:26:48.932118 PDT,"[""mod""]","""lts-end-staff""","""RtxBB""","""urnyxqT1""","""LTS Team""","""URYF1ZY1""","""LTS End Staff""","""LTS End Staff is a utility mod…","[""adventure"", ""forge"", … ""transportation""]","[""adventure"", ""forge"", … ""transportation""]","[""1.20.1""]",98,0,"""https://cdn.modrinth.com/data/…","""2026-02-01T00:23:00.419232+00:…","""2026-01-31T23:56:04.540769+00:…","""joRudX4a""","""LicenseRef-All-Rights-Reserved""","""required""","""required""","[""client_and_server""]",[],[],null,1850220
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""8af5da09-b203-4d7e-bcab-e78002…","""minecraft_java_server""","""PikZOK73""",2026-08-25 14:30:15.650740 PDT,"[""minecraft_java_server""]","""aetheriacobblemon""","""Auristite""","""2SgKIxOh""","""Aetheria Cobblemon""","""6C3U8guf""","""Aetheria Cobblemon""","""Welcome to Aetheria, a strange…","[""keep-inventory"", ""pokemon"", … ""survival-mode""]","[""pokemon""]",[],0,1,"""https://cdn.modrinth.com/data/…","""2026-03-02T18:01:20.839185+00:…","""2026-03-02T17:36:06.578604+00:…","""NL9DdrS4""","""LicenseRef-Unknown""","""unknown""","""unknown""",[],[],"[""https://cdn.modrinth.com/data/PikZOK73/images/82f8be0703b6e65fe9b9fd25c65f9f13b5784bf4_350.webp"", ""https://cdn.modrinth.com/data/PikZOK73/images/5f08cd69817e5f033703b6407ee8cfaddcb39e06_350.webp"", ""https://cdn.modrinth.com/data/PikZOK73/images/7f4a8af3d211ea924e5b9e0f8ff4b4db688f14f5_350.webp""]","""https://cdn.modrinth.com/data/…",10473973
"""8af5da09-b203-4d7e-bcab-e78002…","""minecraft_java_server""","""4LXgw0dN""",2026-08-25 14:30:15.650740 PDT,"[""minecraft_java_server""]","""lemoncloud""","""iLemon""","""O8CjwrcY""",null,null,"""LemonCloud Cobblemon""","""Battle your way through Gyms, …","[""bosses"", ""classes"", … ""social""]","[""pokemon"", ""skyblock"", ""smp""]",[],0,4,"""https://cdn.modrinth.com/data/…","""2026-03-02T17:59:50.146210+00:…","""2026-03-02T17:34:56.479932+00:…","""tMrap9ws""","""LicenseRef-Unknown""","""unknown""","""unknown""",[],[],"[""https://cdn.modrinth.com/data/4LXgw0dN/images/31587045a7112f75bb2ada9aa8b6add0e6670c6c.gif""]",null,16048946
